In [7]:
""" 
--> system prompt
--> output structure
--> init agent
--> tools
"""

' \n--> system prompt\n--> output structure\n--> init agent\n--> tools\n'

In [ ]:
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.messages import HumanMessage
from pydantic import BaseModel

### Structure output


In [3]:
# get the format output from the llm
system_prompt = """ 
You are very good ai assistent for write the code and generating the content
"""

class OutputFormate(BaseModel):
    question:str
    description:str 
    marks:str


agent = create_agent(
    model="gpt-5.5",
    system_prompt=system_prompt,
    response_format=OutputFormate
)

question = HumanMessage(content="Generate the random five math question for 12th standard student from linear equation.")

result = agent.invoke({
    "messages":[question]
})

In [5]:
x = result["messages"][-1].content

In [ ]:
'{"question":"1. Solve the system of linear equations: 2x + 3y - z = 5, x - 2y + 4z = 11, 3x + y + 2z = 10.\\n\\n2. Find the value of k for which the system of equations x + y + z = 6, 2x + 3y + z = 10, x + 2y + kz = 9 has a unique solution.\\n\\n3. Solve the following pair of linear equations using matrix method: 3x - 2y = 7 and 5x + y = 9.\\n\\n4. Determine whether the following system is consistent or inconsistent: x + 2y + 3z = 4, 2x + 4y + 6z = 8, 3x + 6y + 9z = 15.\\n\\n5. A shopkeeper sells two types of notebooks. The cost of 3 notebooks of type A and 2 notebooks of type B is ₹190. The cost of 2 notebooks of type A and 5 notebooks of type B is ₹310. Find the cost of each type of notebook.","description":"Five random linear equation questions suitable for Class 12 students. These questions cover solving systems of linear equations, matrix method, consistency of equations, parameter-based conditions, and word problems.","marks":"5 marks each"}'

### Tools 
#### Mini-project

In [18]:
from langchain.tools import tool
from langchain.messages import HumanMessage
from datetime import datetime

@tool
def get_local_time():
    """ 
    The function use to get the local time.
    It return the integer value which is current hour.
    """
    hour = datetime.now().hour 
    return hour 



In [19]:
system_prompt = """ 
You are a AI assistant for user. Help in daily task and make it easy.
"""

agent = create_agent(
    model="gpt-5.5",
    system_prompt=system_prompt,
    tools=[get_local_time]
)

In [20]:
result = agent.invoke({
    "messages":[HumanMessage(content="Get the current time and remind me that its time for bed or not ?. I usually go bed at 20pm.")]
})

In [21]:
result["messages"][-1].content

'It’s currently **19:00 (7 PM)**.\n\nYou usually go to bed at **20:00 (8 PM)**, so **it’s not bedtime yet**. You have about **1 hour** before bed.'

In [ ]:
'It’s currently **19:00 (7 PM)**.\n\nYou usually go to bed at **20:00 (8 PM)**, so **it’s not bedtime yet**. You have about **1 hour** before bed.'

#### Multiple tools 

In [32]:
@tool
def get_user(user_id:int):
    """ Get the user with the help of user id if user_id is in table return username else return message """
    data = [
    {
        "user_id": 101,
        "email": "apple@gmail.com",
        "username": "rairai"
    },
    {
        "user_id": 102,
        "email": "alex.morgan@gmail.com",
        "username": "alexm"
    },
    {
        "user_id": 103,
        "email": "sophia.chen@yahoo.com",
        "username": "sophiac"
    },
    {
        "user_id": 104,
        "email": "david.miller@outlook.com",
        "username": "dmiller99"
    },
    {
        "user_id": 105,
        "email": "emma.watson@gmail.com",
        "username": "emma_w"
    }
    ]

    for  user_data in data:
        if user_data["user_id"] == user_id:
            return user_data["username"]
    return "No user is found"


@tool
def get_order(username:str):
    """ check the product table with username if username found return order else return simple message """
    order = [{
        "uesrname":"rairai",
        "order":["laptop","airbuds"]
    }]
    if username in order[0].values():
        return order[0]["order"]
    else:
        return "No order is placed with this username"



In [33]:
system_prompt = """ 
You are very good customer service assistent. Help user to resolve the query
"""

agent = create_agent(
    model="gpt-5.5",
    system_prompt=system_prompt,
    tools=[get_user,get_order]
)

In [28]:
human_message = HumanMessage(content="Who are you ?")
agent.invoke({
    "messages":[human_message]
})

{'messages': [HumanMessage(content='Who are you ?', additional_kwargs={}, response_metadata={}, id='0b688232-0cfe-4a1e-8051-6bee584f8608'),
  AIMessage(content='I’m your customer service assistant. I’m here to help answer questions, look up information when needed, and guide you through resolving any issue.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 193, 'total_tokens': 225, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-EehMwZU0PHwnqAmZIJdBK4nIYj9HW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18d18-37f9-7596-bfbb-71be

In [34]:
human_message = HumanMessage(content="Get the order of 101 user id")
response = agent.invoke({
    "messages":[human_message]
})

In [35]:
print(response["messages"][-1].content)

User ID 101 belongs to **rairai**.

Orders found:
- laptop
- airbuds
